In [1]:
# Cell 1 - Import Libraries

import pandas as pd
import numpy as np
import joblib
import json
import os
import warnings
warnings.filterwarnings("ignore")

print("Libraries imported")
print("Pandas     :", pd.__version__)
print("Numpy      :", np.__version__)
print("Joblib     :", joblib.__version__)

Libraries imported
Pandas     : 2.3.3
Numpy      : 2.3.5
Joblib     : 1.5.2


In [2]:
# Cell 2 - Load Models

rain_model = joblib.load(r"C:\weather-ai-project\models\rain_classifier.pkl")
temp_model = joblib.load(r"C:\weather-ai-project\models\temp_regressor.pkl")
scaler = joblib.load(r"C:\weather-ai-project\models\scaler.pkl")

with open(r"C:\weather-ai-project\models\reg_columns.json", "r") as f:
    reg_columns = json.load(f)

print("Models loaded successfully")
print("Rain Classifier  :", type(rain_model).__name__)
print("Temp Regressor   :", type(temp_model).__name__)
print("Scaler           :", type(scaler).__name__)
print("Reg Columns      :", len(reg_columns), "columns")

Models loaded successfully
Rain Classifier  : XGBClassifier
Temp Regressor   : XGBRegressor
Scaler           : StandardScaler
Reg Columns      : 23 columns


In [3]:
# Cell 3 - Load Data for Reference

df = pd.read_csv(r"C:\weather-ai-project\data\weather_preprocessed.csv")

print("Data loaded successfully")
print("Shape      :", df.shape)
print("Columns    :", list(df.columns))

Data loaded successfully
Shape      : (142193, 29)
Columns    : ['Location', 'MinTemp', 'MaxTemp', 'Rainfall', 'Evaporation', 'Sunshine', 'WindGustDir', 'WindGustSpeed', 'WindDir9am', 'WindDir3pm', 'WindSpeed9am', 'WindSpeed3pm', 'Humidity9am', 'Humidity3pm', 'Pressure9am', 'Pressure3pm', 'Cloud9am', 'Cloud3pm', 'Temp9am', 'Temp3pm', 'RainToday', 'RainTomorrow', 'Year', 'Month', 'Season', 'Temp_Range', 'Humidity_Drop', 'Pressure_Drop', 'Wind_Change']


In [4]:
# Cell 4 - Recommendation Logic by Industry

def get_recommendations(rain_prob, max_temp, wind_speed, humidity):

    recommendations = {}

    # Agriculture
    if rain_prob >= 0.61:
        recommendations["Agriculture"] = "Avoid spraying pesticides. Good day for irrigation planning."
    elif max_temp > 40:
        recommendations["Agriculture"] = "Water crops early morning. Risk of heat stress on plants."
    else:
        recommendations["Agriculture"] = "Safe for plowing, planting and field work."

    # Construction
    if wind_speed > 120:
        recommendations["Construction"] = "Halt all outdoor construction. Dangerous wind conditions."
    elif rain_prob >= 0.61:
        recommendations["Construction"] = "Avoid concrete pouring and roofing work today."
    elif max_temp > 40:
        recommendations["Construction"] = "Limit outdoor work to early hours. Provide shade and water."
    else:
        recommendations["Construction"] = "Safe for all outdoor construction activities."

    # Transportation
    if rain_prob >= 0.61:
        recommendations["Transportation"] = "Allow extra travel time. Roads may be slippery."
    elif wind_speed > 100:
        recommendations["Transportation"] = "Caution for high-sided vehicles. Avoid open highways."
    else:
        recommendations["Transportation"] = "Normal travel conditions. No major disruptions expected."

    # Tourism
    if rain_prob >= 0.61:
        recommendations["Tourism"] = "Plan indoor attractions. Carry rain gear if going outside."
    elif max_temp > 40:
        recommendations["Tourism"] = "Avoid midday outdoor tours. Risk of heat exhaustion."
    else:
        recommendations["Tourism"] = "Excellent day for outdoor sightseeing and activities."

    # Energy
    if humidity > 80 and rain_prob >= 0.61:
        recommendations["Energy"] = "Low solar output expected. Wind energy may increase."
    elif max_temp > 38:
        recommendations["Energy"] = "High electricity demand expected. Peak load alert."
    else:
        recommendations["Energy"] = "Stable energy conditions. Good solar generation expected."

    # Logistics
    if rain_prob >= 0.61 or wind_speed > 100:
        recommendations["Logistics"] = "Expect delivery delays. Reschedule non-urgent shipments."
    else:
        recommendations["Logistics"] = "Clear conditions for deliveries and warehouse operations."

    return recommendations

# Test the function
test = get_recommendations(rain_prob=0.75, max_temp=35, wind_speed=60, humidity=85)
for industry, advice in test.items():
    print(f"{industry:15} : {advice}")

Agriculture     : Avoid spraying pesticides. Good day for irrigation planning.
Construction    : Avoid concrete pouring and roofing work today.
Transportation  : Allow extra travel time. Roads may be slippery.
Tourism         : Plan indoor attractions. Carry rain gear if going outside.
Energy          : Low solar output expected. Wind energy may increase.
Logistics       : Expect delivery delays. Reschedule non-urgent shipments.


In [5]:
# Cell 5 - Alert Logic (Low / Medium / High)

def get_alert_level(rain_prob, max_temp, wind_speed, rainfall):

    alerts = []
    level = "Low"

    # Heat Alert
    if max_temp > 40:
        alerts.append("HEAT ALERT — Extreme temperature above 40C")
        level = "High"
    elif max_temp > 35:
        alerts.append("HEAT WARNING — Temperature above 35C")
        if level == "Low":
            level = "Medium"

    # Flood Alert
    if rain_prob >= 0.80:
        alerts.append("FLOOD ALERT — Very high rain probability")
        level = "High"
    elif rain_prob >= 0.61:
        alerts.append("RAIN WARNING — Moderate to high rain probability")
        if level == "Low":
            level = "Medium"

    # Storm Alert
    if wind_speed > 120:
        alerts.append("STORM ALERT — Wind speed above 120 km/h")
        level = "High"
    elif wind_speed > 90:
        alerts.append("WIND WARNING — Strong winds above 90 km/h")
        if level == "Low":
            level = "Medium"

    # Heavy Rainfall Alert
    if rainfall > 80:
        alerts.append("HEAVY RAINFALL ALERT — Rainfall above 80mm")
        level = "High"

    # No alerts
    if len(alerts) == 0:
        alerts.append("No active weather alerts")

    return level, alerts

# Test the function
level, alerts = get_alert_level(rain_prob=0.85, max_temp=42, wind_speed=130, rainfall=90)
print("Alert Level :", level)
print("Alerts      :")
for a in alerts:
    print("  -", a)

Alert Level : High
Alerts      :
  - HEAT ALERT — Extreme temperature above 40C
  - FLOOD ALERT — Very high rain probability
  - STORM ALERT — Wind speed above 120 km/h
  - HEAVY RAINFALL ALERT — Rainfall above 80mm


In [6]:
# Cell 6 - ML Prediction Function (Fixed Final)

def get_ml_prediction(input_dict):

    cls_columns = [col for col in df.columns if col != "RainTomorrow"]
    cls_input = pd.DataFrame([input_dict])[cls_columns]

    scale_cols = ["MinTemp", "MaxTemp", "Rainfall", "Evaporation", "Sunshine",
                  "WindGustSpeed", "WindSpeed9am", "WindSpeed3pm",
                  "Humidity9am", "Humidity3pm", "Pressure9am", "Pressure3pm",
                  "Cloud9am", "Cloud3pm", "Temp9am", "Temp3pm"]

    cls_input[scale_cols] = scaler.transform(cls_input[scale_cols])

    rain_prob = rain_model.predict_proba(cls_input)[0][1]
    rain_pred = 1 if rain_prob >= 0.61 else 0

    reg_input = pd.DataFrame([input_dict])[reg_columns]
    reg_scale_cols = [col for col in scale_cols if col in reg_columns]
    scaled_all = scaler.transform(pd.DataFrame([input_dict])[scale_cols])
    col_indices = [scale_cols.index(c) for c in reg_scale_cols]
    reg_input[reg_scale_cols] = scaled_all[:, col_indices]

    temp_pred_scaled = temp_model.predict(reg_input)[0]

    # Inverse transform temperature
    dummy = np.zeros((1, len(scale_cols)))
    dummy[0][scale_cols.index("MaxTemp")] = temp_pred_scaled
    temp_pred_actual = scaler.inverse_transform(dummy)[0][scale_cols.index("MaxTemp")]

    return {
        "rain_prob" : round(float(rain_prob) * 100, 2),
        "rain_pred" : "Yes" if rain_pred == 1 else "No",
        "temp_pred" : round(float(temp_pred_actual), 2)
    }

# Test with raw realistic values
raw_sample = {
    "Location": 10, "MinTemp": 18.0, "MaxTemp": 32.0, "Rainfall": 5.0,
    "Evaporation": 6.0, "Sunshine": 7.0, "WindGustDir": 5, "WindGustSpeed": 55.0,
    "WindDir9am": 3, "WindDir3pm": 7, "WindSpeed9am": 20.0, "WindSpeed3pm": 28.0,
    "Humidity9am": 70.0, "Humidity3pm": 60.0, "Pressure9am": 1015.0,
    "Pressure3pm": 1012.0, "Cloud9am": 4.0, "Cloud3pm": 5.0,
    "Temp9am": 22.0, "Temp3pm": 29.0, "RainToday": 0,
    "Year": 2015, "Month": 6, "Season": 2,
    "Temp_Range": 14.0, "Humidity_Drop": 10.0,
    "Pressure_Drop": 3.0, "Wind_Change": 8.0, "RainTomorrow": 0
}

result = get_ml_prediction(raw_sample)
print("Rain Probability :", result["rain_prob"], "%")
print("Rain Prediction  :", result["rain_pred"])
print("Temp Prediction  :", result["temp_pred"], "C")

Rain Probability : 60.39 %
Rain Prediction  : No
Temp Prediction  : 27.19 C


In [9]:
# Cell 7 - Save Utility Functions to utils folder

utils_code = '''
import numpy as np
import pandas as pd
import joblib
import json

# Load models
rain_model  = joblib.load(r"C:\\weather-ai-project\\models\\rain_classifier.pkl")
temp_model  = joblib.load(r"C:\\weather-ai-project\\models\\temp_regressor.pkl")
scaler      = joblib.load(r"C:\\weather-ai-project\\models\\scaler.pkl")

with open(r"C:\\weather-ai-project\\models\\reg_columns.json", "r") as f:
    reg_columns = json.load(f)

SCALE_COLS = ["MinTemp", "MaxTemp", "Rainfall", "Evaporation", "Sunshine",
              "WindGustSpeed", "WindSpeed9am", "WindSpeed3pm",
              "Humidity9am", "Humidity3pm", "Pressure9am", "Pressure3pm",
              "Cloud9am", "Cloud3pm", "Temp9am", "Temp3pm"]

def get_ml_prediction(input_dict):
    cls_columns = list(input_dict.keys())
    cls_input   = pd.DataFrame([input_dict])

    cls_input[SCALE_COLS] = scaler.transform(cls_input[SCALE_COLS])

    rain_prob = rain_model.predict_proba(cls_input)[0][1]
    rain_pred = 1 if rain_prob >= 0.61 else 0

    reg_input      = pd.DataFrame([input_dict])[reg_columns]
    reg_scale_cols = [col for col in SCALE_COLS if col in reg_columns]
    scaled_all     = scaler.transform(pd.DataFrame([input_dict])[SCALE_COLS])
    col_indices    = [SCALE_COLS.index(c) for c in reg_scale_cols]
    reg_input[reg_scale_cols] = scaled_all[:, col_indices]

    temp_pred_scaled = temp_model.predict(reg_input)[0]
    dummy            = np.zeros((1, len(SCALE_COLS)))
    dummy[0][SCALE_COLS.index("MaxTemp")] = temp_pred_scaled
    temp_pred_actual = scaler.inverse_transform(dummy)[0][SCALE_COLS.index("MaxTemp")]

    return {
        "rain_prob" : round(float(rain_prob) * 100, 2),
        "rain_pred" : "Yes" if rain_pred == 1 else "No",
        "temp_pred" : round(float(temp_pred_actual), 2)
    }

def get_recommendations(rain_prob, max_temp, wind_speed, humidity):
    recommendations = {}

    if rain_prob >= 61:
        recommendations["Agriculture"] = "Avoid spraying pesticides. Good day for irrigation planning."
    elif max_temp > 40:
        recommendations["Agriculture"] = "Water crops early morning. Risk of heat stress on plants."
    else:
        recommendations["Agriculture"] = "Safe for plowing, planting and field work."

    if wind_speed > 120:
        recommendations["Construction"] = "Halt all outdoor construction. Dangerous wind conditions."
    elif rain_prob >= 61:
        recommendations["Construction"] = "Avoid concrete pouring and roofing work today."
    elif max_temp > 40:
        recommendations["Construction"] = "Limit outdoor work to early hours. Provide shade and water."
    else:
        recommendations["Construction"] = "Safe for all outdoor construction activities."

    if rain_prob >= 61:
        recommendations["Transportation"] = "Allow extra travel time. Roads may be slippery."
    elif wind_speed > 100:
        recommendations["Transportation"] = "Caution for high-sided vehicles. Avoid open highways."
    else:
        recommendations["Transportation"] = "Normal travel conditions. No major disruptions expected."

    if rain_prob >= 61:
        recommendations["Tourism"] = "Plan indoor attractions. Carry rain gear if going outside."
    elif max_temp > 40:
        recommendations["Tourism"] = "Avoid midday outdoor tours. Risk of heat exhaustion."
    else:
        recommendations["Tourism"] = "Excellent day for outdoor sightseeing and activities."

    if humidity > 80 and rain_prob >= 61:
        recommendations["Energy"] = "Low solar output expected. Wind energy may increase."
    elif max_temp > 38:
        recommendations["Energy"] = "High electricity demand expected. Peak load alert."
    else:
        recommendations["Energy"] = "Stable energy conditions. Good solar generation expected."

    if rain_prob >= 61 or wind_speed > 100:
        recommendations["Logistics"] = "Expect delivery delays. Reschedule non-urgent shipments."
    else:
        recommendations["Logistics"] = "Clear conditions for deliveries and warehouse operations."

    return recommendations

def get_alert_level(rain_prob, max_temp, wind_speed, rainfall):
    alerts = []
    level  = "Low"

    if max_temp > 40:
        alerts.append("HEAT ALERT — Extreme temperature above 40C")
        level = "High"
    elif max_temp > 35:
        alerts.append("HEAT WARNING — Temperature above 35C")
        if level == "Low":
            level = "Medium"

    if rain_prob >= 80:
        alerts.append("FLOOD ALERT — Very high rain probability")
        level = "High"
    elif rain_prob >= 61:
        alerts.append("RAIN WARNING — Moderate to high rain probability")
        if level == "Low":
            level = "Medium"

    if wind_speed > 120:
        alerts.append("STORM ALERT — Wind speed above 120 km/h")
        level = "High"
    elif wind_speed > 90:
        alerts.append("WIND WARNING — Strong winds above 90 km/h")
        if level == "Low":
            level = "Medium"

    if rainfall > 80:
        alerts.append("HEAVY RAINFALL ALERT — Rainfall above 80mm")
        level = "High"

    if len(alerts) == 0:
        alerts.append("No active weather alerts")

    return level, alerts
'''

utils_path = r"C:\weather-ai-project\utils\ml_utils.py"
with open(utils_path, "w") as f:
    f.write(utils_code)

print("ml_utils.py saved successfully")
print("Path :", utils_path)

ml_utils.py saved successfully
Path : C:\weather-ai-project\utils\ml_utils.py


In [10]:
# Cell 9 - Write requirements.txt

requirements = """streamlit
pandas
numpy
scikit-learn
xgboost
joblib
requests
matplotlib
seaborn
plotly
"""

req_path = r"C:\weather-ai-project\requirements.txt"
with open(req_path, "w") as f:
    f.write(requirements)

print("requirements.txt saved successfully")
print("Path :", req_path)

requirements.txt saved successfully
Path : C:\weather-ai-project\requirements.txt


In [11]:
# Cell 10 - Write .streamlit/config.toml

config = """
[theme]
primaryColor             = "#00b4d8"
backgroundColor          = "#0d1b2a"
secondaryBackgroundColor = "#1b2838"
textColor                = "#ffffff"
font                     = "sans serif"

[server]
headless                 = true
enableCORS               = false
"""

config_path = r"C:\weather-ai-project\.streamlit\config.toml"
with open(config_path, "w") as f:
    f.write(config)

print("config.toml saved successfully")
print("Theme  : Ocean Teal")
print("Path   :", config_path)

config.toml saved successfully
Theme  : Ocean Teal
Path   : C:\weather-ai-project\.streamlit\config.toml


In [12]:
# Cell 11 - Write secrets.toml Template (placeholder only)

secrets = """
[general]
OPENWEATHER_API_KEY = "YOUR_API_KEY_HERE"
"""

secrets_path = r"C:\weather-ai-project\.streamlit\secrets.toml"
with open(secrets_path, "w") as f:
    f.write(secrets)

print("secrets.toml saved successfully")
print("Path     :", secrets_path)
print("")
print("IMPORTANT — Open this file and replace:")
print("  YOUR_API_KEY_HERE")
print("  with your real OpenWeather API key")
print("")
print("DO NOT upload this file to GitHub")

secrets.toml saved successfully
Path     : C:\weather-ai-project\.streamlit\secrets.toml

IMPORTANT — Open this file and replace:
  c375e85e7a6fe8c8d6297a9c161930ca
  with your real OpenWeather API key

DO NOT upload this file to GitHub


In [13]:
# Cell 12 - Write .gitignore

gitignore = """
.streamlit/secrets.toml
data/
__pycache__/
*.pyc
.env
"""

gitignore_path = r"C:\weather-ai-project\.gitignore"
with open(gitignore_path, "w") as f:
    f.write(gitignore)

print(".gitignore saved successfully")
print("Path :", gitignore_path)
print("")
print("Blocked from GitHub :")
print("  - secrets.toml")
print("  - data folder")
print("  - pycache")

.gitignore saved successfully
Path : C:\weather-ai-project\.gitignore

Blocked from GitHub :
  - secrets.toml
  - data folder
  - pycache


In [14]:
# Cell 13 - Final Folder Structure Check

import os

base = r"C:\weather-ai-project"

for root, dirs, files in os.walk(base):
    dirs[:] = [d for d in dirs if d not in ["data", "__pycache__"]]

    level    = root.replace(base, "").count(os.sep)
    indent   = "    " * level
    folder   = os.path.basename(root)
    print(f"{indent}{folder}/")

    subindent = "    " * (level + 1)
    for file in files:
        size     = os.path.getsize(os.path.join(root, file))
        size_str = f"{round(size/1024, 1)} KB"
        print(f"{subindent}{file}  ({size_str})")

weather-ai-project/
    .gitignore  (0.1 KB)
    app.py  (0.0 KB)
    requirements.txt  (0.1 KB)
    .streamlit/
        config.toml  (0.3 KB)
        secrets.toml  (0.1 KB)
    assets/
        actual_vs_predicted.png  (187.1 KB)
        confusion_matrix.png  (42.9 KB)
        feature_importance_cls.png  (64.9 KB)
        feature_importance_reg.png  (57.0 KB)
        model_comparison.png  (113.4 KB)
        precision_recall_curve.png  (40.0 KB)
        residual_plot.png  (245.3 KB)
        roc_curve.png  (55.4 KB)
        threshold_analysis.png  (57.5 KB)
    dashboard/
    models/
        rain_classifier.pkl  (442.0 KB)
        reg_columns.json  (0.3 KB)
        scaler.pkl  (1.4 KB)
        temp_regressor.pkl  (481.7 KB)
    notebooks/
    utils/
        ml_utils.py  (5.1 KB)
    WEATHER AI ANALYSIS/
        .gitattributes  (0.1 KB)
        .git/
            COMMIT_EDITMSG  (0.0 KB)
            config  (0.2 KB)
            description  (0.1 KB)
            HEAD  (0.0 KB)
            i